# Loop Corridor Radius Sensitivity

The corridor radius $r = 100 \text{m}$ is a modeling choice based on what a cyclist experiences along a loop. We here recompute the loop land use shares, components, and main-specification score for other radii. We also check how much the loop ranking changes. The goal is to see whether the results depend on the choice of $r$.

Note that running this notebook requires that `pixi run loop-score` has been run once for each of these settings in [src/splendor/config.py](../../src/splendor/config.py):
- `CORRIDOR_RADIUS = 50`
- `CORRIDOR_RADIUS = 100` (the default)
- `CORRIDOR_RADIUS = 200`

Also note that this is a copy of [notebooks/exp/link_corridor_radius.ipynb](../../notebooks/exp/link_corridor_radius.ipynb) with the level changed to loops.

<br><br>

## 0. Setup

### 0.1. Imports and Config

In [1]:
import pandas as pd

from splendor.config import CORRIDOR_RADIUS, ROAD_STRIP_RADIUS, STUDY_AREA
from splendor.landuse.shares import SHARE_PREFIX, get_share_type_id
from splendor.landuse.types import SHORT_NAMES
from splendor.paths import landuse_path, components_path, splendor_path

LEVEL = "loops"
ID_COLUMN = "loop_id"
RADII = (50, 100, 200)
REFERENCE_RADIUS = CORRIDOR_RADIUS
TOP_N = 20

assert REFERENCE_RADIUS in RADII, "CORRIDOR_RADIUS in config is not among the compared radii"

### 0.2. Data

In [2]:
def load_by_radius(path_function) -> dict[int, pd.DataFrame]:
    return {
        r: pd.read_csv(path_function(STUDY_AREA, LEVEL, r, ROAD_STRIP_RADIUS), index_col=ID_COLUMN)
        for r in RADII
    }

landuse = load_by_radius(landuse_path)
components = load_by_radius(components_path)
splendor = load_by_radius(splendor_path)

for frames in (landuse, components, splendor):
    ids = frames[REFERENCE_RADIUS].index
    assert all(df.index.equals(ids) for df in frames.values()), "edge ids differ between radii"

print(f"{len(ids)} loops, radii {RADII}")

45218 loops, radii (50, 100, 200)


<br><br>

## 1. Land Use Shares per Radius

In [3]:
share_columns = [c for c in landuse[REFERENCE_RADIUS].columns if c.startswith(SHARE_PREFIX)]

# Mean share per type over loops
mean_shares = pd.DataFrame({r: df[share_columns].mean() for r, df in landuse.items()})
mean_shares.index = [SHORT_NAMES[get_share_type_id(c)] for c in share_columns]
print(mean_shares.sort_values(REFERENCE_RADIUS, ascending=False).round(3))

# Corridor size
print(pd.DataFrame({r: df["n_pixels"].describe() for r, df in landuse.items()}).round(0))

               50     100    200
Cropland     0.613  0.650  0.684
Settlement   0.192  0.156  0.119
Forest       0.092  0.089  0.088
Road/rail    0.059  0.053  0.049
Grassland    0.023  0.023  0.023
Lakes        0.005  0.008  0.013
Mires        0.005  0.007  0.008
Rivers       0.006  0.006  0.007
Sea          0.003  0.005  0.007
Coast        0.003  0.002  0.002
Lagoons      0.000  0.000  0.000
Heathland    0.000  0.000  0.000
Sparse veg.  0.000  0.000  0.000
           50       100       200
count  45218.0  45218.0   45218.0
mean   30131.0  64475.0  132299.0
std     3900.0   8317.0   17071.0
min     2003.0   4385.0    9130.0
25%    28745.0  61525.0  126259.0
50%    31322.0  67067.0  137645.0
75%    32802.0  70205.0  144046.0
max    34677.0  73826.0  152040.0


<br><br>

## 2. Components per Radius

In [4]:
component_columns = list(components[REFERENCE_RADIUS].columns)

summary = pd.concat(
    {r: df[component_columns].describe().T[["mean", "std", "min", "50%", "max"]] for r, df in components.items()},
    names=["r", "component"],
)
print(summary.swaplevel().sort_index().round(3))

                         mean    std    min    50%    max
component          r                                     
attractiveness     50   0.223  0.025  0.018  0.223  0.551
                   100  0.236  0.024  0.036  0.236  0.525
                   200  0.251  0.025  0.074  0.251  0.513
diversity_richness 50   2.483  0.588  1.000  2.000  5.000
                   100  2.322  0.609  1.000  2.000  5.000
                   200  1.979  0.738  1.000  2.000  5.000
diversity_shannon  50   1.150  0.194  0.546  1.141  1.891
                   100  1.114  0.212  0.491  1.105  1.910
                   200  1.077  0.231  0.420  1.067  1.865


<br><br>

## 3. Rank Correlations

In [5]:
def get_reference_correlations(frames: dict[int, pd.DataFrame], columns: list[str]) -> pd.DataFrame:
    reference = frames[REFERENCE_RADIUS]
    return pd.DataFrame(
        {r: [df[c].corr(reference[c], method="spearman") for c in columns]
         for r, df in frames.items() if r != REFERENCE_RADIUS},
        index=columns,
    )

correlations = pd.concat([
    get_reference_correlations(components, component_columns),
    get_reference_correlations(splendor, ["score"]),
])
print(correlations.round(3))

# Main score, all pairs
scores = pd.DataFrame({r: df["score"] for r, df in splendor.items()})
print(scores.corr(method="spearman").round(3))

                      50     200
diversity_shannon   0.994  0.992
diversity_richness  0.798  0.720
attractiveness      0.978  0.954
score               0.992  0.987
       50     100    200
50   1.000  0.992  0.964
100  0.992  1.000  0.987
200  0.964  0.987  1.000


<br><br>

## 4. Top and Bottom Loops

In [6]:
ranks = pd.DataFrame({r: df["rank"] for r, df in splendor.items()}) # Rank 1 is the best loop

def get_extreme_ids(rank: pd.Series, n: int, which: str) -> set:
    return set((rank.nsmallest(n) if which == "top" else rank.nlargest(n)).index)

overlap = pd.DataFrame({
    which: {
        r: len(get_extreme_ids(ranks[r], TOP_N, which) & get_extreme_ids(ranks[REFERENCE_RADIUS], TOP_N, which))
        for r in RADII if r != REFERENCE_RADIUS
    }
    for which in ("top", "bottom")
})
print(f"Loops shared with the {TOP_N} at r = {REFERENCE_RADIUS} m:")
print(overlap)

# Where the reference top loops rank at the other radii
top_ids = ranks[REFERENCE_RADIUS].nsmallest(TOP_N).index
print(ranks.loc[top_ids].sort_values(REFERENCE_RADIUS).astype(int))

Loops shared with the 20 at r = 100 m:
     top  bottom
50    17      18
200   19      16
         50   100  200
loop_id               
27083      1    1    1
27085      2    2    2
30233      3    3    3
26989      4    4    4
26796      8    5    5
27065      6    6    6
30231      5    7    9
26987      7    8    8
26049     12    9    7
26892     11   10   10
27082     10   11   11
27076      9   12   12
26799     18   13   14
26069     17   14   13
30240     13   15   16
26904     15   16   19
26050     23   17   15
26976     19   18   20
26075     22   19   17
26797     24   20   22


<br><br>

## 5. Conclusion

The loop ranking is robust to the corridor radius. The main score score correlates 0.99 with both $r = 50$ and $r = 200$, and the correlation between the two extremes is 0.96. Of the top 20 loops at $r = 100$, 17 ($r = 50$) and 19 ($r = 200$) stay. Of the bottom 20, 18 and 16 stay. Shannon entropy and attractiveness are also stable (0.992-0.994). Richness is less so (0.72-0.80), but that isn't part of the main specification. 

Larger radii shift the shares from settlemtn and road/rail toward cropland, since settlement concentrates near the road. Diversity therefore decreases slightly and attractiveness increases slightly.

We therefore keep $r = 100 \text{m}$.